In [ ]:
"""A customer made 3 transactions worth Rs 85,000 in 8 minutes across 3 different merchants. Their total daily transaction count looks normal.
 Your GROUP BY pipeline shows nothing wrong. The fraud team only spots it 4 days later by looking at individual records.

The fraud signal: 3 or more transactions within any 10-minute window where the amount exceeds 3x the customer's own historical average.
 GROUP BY can count transactions. It cannot see time gaps between them.
"""
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum,count

# ── Create Spark Session ─────────────────────────────
spark = SparkSession.builder \
    .appName("Skew Handling with Salting") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# ── Read Data ───────────────────────────────────────
df = spark.read.option("mergeSchema", "true").parquet(
    r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions"
)
# Fraud detection mistake
df.groupBy("transaction_id").agg(count("*").alias("total_tran"),sum("amount").alias("total_amount")).show()

+--------------+----------+------------+
|transaction_id|total_tran|total_amount|
+--------------+----------+------------+
|   T0000560437|         1|     16833.1|
|   T0000536197|         1|     3950.64|
|   T0000658480|         1|    35911.32|
|   T0000796539|         1|     1242.44|
|   T0000273539|         1|      269.17|
|   T0000279694|         1|       320.2|
|   T0000231122|         1|     1754.09|
|   T0000438524|         1|    12435.02|
|   T0000053616|         1|     8023.61|
|   T0000772454|         1|     3053.07|
|   T0000958399|         1|      2618.7|
|   T0000349327|         1|     4464.12|
|   T0000355599|         1|      4168.2|
|   T0000753877|         1|     7215.34|
|   T0000670612|         1|     1532.44|
|   T0000426727|         1|     9728.49|
|   T0000964689|         1|    29818.12|
|   T0000594063|         1|     1369.13|
|   T0000986275|         1|     2079.24|
|   T0000517407|         1|      743.73|
+--------------+----------+------------+
only showing top

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *
spark = SparkSession.builder \
    .appName("Fraud Detection") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.parquet(
    r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions"
)
# Fix schema inconsistency
df = df.withColumn(
    "amount",
    coalesce(col("amount"), col("transaction_amount"))
)
# Remove null timestamps BEFORE processing
df = df.filter(col("transaction_ts").isNotNull())
# Convert timestamp
df = df.withColumn("ts", unix_timestamp("transaction_ts").cast("long"))
# Windows
w_time = Window.partitionBy("customer_id").orderBy("ts").rangeBetween(-600, 0)
w_cust = Window.partitionBy("customer_id")

# Avg
df = df.withColumn("avg_amt", avg("amount").over(w_cust))
# High flag
df = df.withColumn(
    "high_flag",
    when(col("amount") > col("avg_amt") * 3, 1).otherwise(0)
)
# Count high txns
df = df.withColumn(
    "high_txn_count",
    sum("high_flag").over(w_time)
)
# Fraud detection
fraud = df.filter(col("high_txn_count") >= 3)
fraud.show(truncate=False)

+--------------+-----------+-----------+---------+---------+-------+--------------+-------------------+------------------+----------+---------------+------------------+---------+--------------+
|transaction_id|customer_id|merchant_id|   amount|   status| region|payment_method|     transaction_ts|transaction_amount|        ts|txn_count_10min|           avg_amt|high_flag|high_txn_count|
+--------------+-----------+-----------+---------+---------+-------+--------------+-------------------+------------------+----------+---------------+------------------+---------+--------------+
|   T0001382975|   C0004935|     M00006| 53281.15|completed|   West|   Credit Card|2024-02-17 20:58:06|              NULL|1708203486|              3| 16967.61393939394|        1|             3|
|   T0001668278|   C0004935|     M00317| 53281.15| refunded|   West|   Credit Card|2024-02-17 20:58:52|              NULL|1708203532|              4| 16967.61393939394|        1|             4|
|   T0001953270|   C0005907|  